# Multi-turn Sessions — fasteval-langgraph

Test multi-turn conversations with shared state, conversation metrics, and human-in-the-loop interrupt/resume flows.

**What you'll learn:**

1. **Sessions** — `.session()` with shared `thread_id`
2. **Conversation metrics** — `context_retention`, `consistency`, `topic_drift`
3. **Session history** — Inspect and use `s.history`
4. **State seeding** — Pre-load state with `update_state()`
5. **Interrupt/resume** — Human-in-the-loop testing
6. **Test suite design** — Organizing multi-turn tests

Every cell is **runnable**. Conversation metrics require an OpenAI API key.

---

## Setup

In [ ]:
!pip install -q fasteval-langgraph

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab Secrets.")
except (ImportError, Exception):
    if os.environ.get("OPENAI_API_KEY"):
        print("API key found in environment.")
    else:
        print(
            "WARNING: OPENAI_API_KEY not set.\n"
            "Conversation metrics (context_retention, consistency, topic_drift) will fail.\n"
            "Session mechanics and interrupt/resume still work."
        )

In [ ]:
import fasteval as fe
from fasteval_langgraph import harness

print("Ready.")

---

## Build Test Agents

We'll create two agents: an echo bot (for session mechanics) and an interrupt agent (for human-in-the-loop).

> **Collab Note:** These are deterministic agents for reproducibility. In production, you'd test with your real LLM-powered agent.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
from langchain_core.messages import AIMessage, HumanMessage


# --- Echo bot (accumulates call_count across turns) ---
class EchoState(MessagesState):
    call_count: int


def echo(state: EchoState) -> dict:
    last_msg = state["messages"][-1].content
    return {
        "messages": [AIMessage(content=f"Echo: {last_msg}")],
        "call_count": state.get("call_count", 0) + 1,
    }


eb = StateGraph(EchoState)
eb.add_node("echo", echo)
eb.add_edge(START, "echo")
eb.add_edge("echo", END)
echo_compiled = eb.compile(checkpointer=MemorySaver())


# --- Interrupt agent (human-in-the-loop) ---
class ApprovalState(MessagesState):
    approved: bool


def ask(state):
    return {"messages": [AIMessage(content="Do you approve this action?")]}


def wait(state):
    answer = interrupt("Waiting for user approval")
    return {"approved": answer == "yes"}


def final(state):
    if state.get("approved"):
        return {"messages": [AIMessage(content="Approved! Proceeding.")]}
    return {"messages": [AIMessage(content="Rejected. Action cancelled.")]}


ib = StateGraph(ApprovalState)
ib.add_node("ask", ask)
ib.add_node("wait", wait)
ib.add_node("final", final)
ib.add_edge(START, "ask")
ib.add_edge("ask", "wait")
ib.add_edge("wait", "final")
ib.add_edge("final", END)
interrupt_compiled = ib.compile(checkpointer=MemorySaver())

print("Echo agent and Interrupt agent built.")

---

## 1. Creating a Session

While `.chat()` creates a new thread per call, `.session()` shares a thread across multiple turns.

> **Collab Note:** Sessions are essential for testing stateful agents. Without them, each `.chat()` call starts fresh and the agent has no memory of previous turns.

In [ ]:
echo_graph = harness(echo_compiled)

async with echo_graph.session() as s:
    r1 = await s.chat("Hello")
    r2 = await s.chat("World")

    # Same thread: call_count accumulates
    print(f"Turn 1: {r1.response}  |  call_count={r1.state['call_count']}")
    print(f"Turn 2: {r2.response}  |  call_count={r2.state['call_count']}")
    assert r1.state["call_count"] == 1
    assert r2.state["call_count"] == 2
    print("\nState accumulates across turns — session shares a thread.")

In [ ]:
# Sessions can be used without the context manager
s = echo_graph.session()
print(f"Thread ID: {s.thread_id}")

r = await s.chat("Without context manager")
print(f"Response: {r.response}")

---

## 2. Conversation Metrics

fasteval provides three conversation-specific LLM-as-judge metrics:

| Metric | What it checks |
|--------|---------------|
| `@fe.context_retention` | Does the agent remember earlier turns? |
| `@fe.consistency` | Does the agent avoid contradictions? |
| `@fe.topic_drift` | Does the agent stay on topic? |

> **Collab Note:** These metrics require `history` to be passed to `fe.score()`. Build the history from the session as the conversation progresses.

In [ ]:
# Context retention: does the agent remember information from earlier turns?

@fe.context_retention(threshold=0.7)
def test_context_retention():
    return fe.score(
        actual_output="Your name is Alice, as you mentioned earlier.",
        expected_output="Alice",
        input="What is my name?",
        history=[
            {"role": "user", "content": "My name is Alice"},
            {"role": "assistant", "content": "Nice to meet you, Alice!"},
        ],
    )


result = test_context_retention()
print(f"Passed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

In [ ]:
# Consistency: does the agent avoid contradicting itself?

@fe.consistency(threshold=0.7)
def test_consistency():
    return fe.score(
        actual_output="The Pro plan costs $49/month, as I mentioned.",
        expected_output="$49/month",
        input="Can you repeat the price of the Pro plan?",
        history=[
            {"role": "user", "content": "How much does the Pro plan cost?"},
            {"role": "assistant", "content": "The Pro plan is $49/month."},
        ],
    )


result = test_consistency()
print(f"Passed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

In [ ]:
# Topic drift: does the agent stay on topic?

@fe.topic_drift(threshold=0.7)
def test_topic_drift():
    return fe.score(
        actual_output="Refunds for returns typically take 5-7 business days to process.",
        expected_output="5-7 business days",
        input="And refund timelines?",
        history=[
            {"role": "user", "content": "Tell me about your return policy"},
            {"role": "assistant", "content": "You can return items within 30 days."},
            {"role": "user", "content": "What about exchanges?"},
            {"role": "assistant", "content": "Exchanges are free for the same item in a different size."},
        ],
    )


result = test_topic_drift()
print(f"Passed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

---

## 3. Session History

The session tracks all `ChatResult` objects in order via `s.history`.

> **Collab Note:** Use `s.history` to build the `history` parameter for `fe.score()` without manually tracking each turn.

In [ ]:
async with echo_graph.session() as s:
    await s.chat("First message")
    await s.chat("Second message")
    await s.chat("Third message")

    print(f"History length: {len(s.history)}")
    for i, result in enumerate(s.history):
        print(f"  Turn {i+1}: {result.response}  |  call_count={result.state['call_count']}")

---

## 4. State Seeding

Use `update_state()` to pre-load data into the graph's checkpoint before chatting.

This is useful for:
- Skipping to a specific conversation state
- Injecting mock data (user prefs, retrieved docs, etc.)
- Testing states that are hard to reach organically

> **Collab Note:** `update_state()` writes to the LangGraph checkpoint, so the next `.chat()` call picks up the seeded values.

In [ ]:
async with echo_graph.session() as s:
    # First, establish the thread with a chat
    r1 = await s.chat("Hello")
    print(f"Before seed: call_count = {r1.state['call_count']}")

    # Continue the conversation
    r2 = await s.chat("After seed")
    print(f"After chat:  call_count = {r2.state['call_count']}")

---

## 5. Interrupt and Resume

For human-in-the-loop agents, the session automatically detects interrupts and uses `resume_fn` on the next `.chat()` call.

> **Collab Note:** The default `resume_fn` sends `Command(resume=msg)`. Override it if your agent expects a different resume format.

In [ ]:
interrupt_graph = harness(interrupt_compiled)

# --- Approval path ---
print("=== Approval Path ===")
async with interrupt_graph.session() as s:
    r1 = await s.chat("Start the process")
    print(f"Turn 1: {r1.response}")
    assert "approve" in r1.response.lower()

    r2 = await s.chat("yes")
    print(f"Turn 2: {r2.response}")
    assert "Approved" in r2.response

# --- Rejection path ---
print("\n=== Rejection Path ===")
async with interrupt_graph.session() as s:
    r1 = await s.chat("Start the process")
    print(f"Turn 1: {r1.response}")

    r2 = await s.chat("no")
    print(f"Turn 2: {r2.response}")
    assert "Rejected" in r2.response

print("\nBoth interrupt paths verified.")

---

## 6. Designing Multi-turn Test Suites

Organize tests by conversation scenario. Each test class covers one flow.

> **Collab Note:** In pytest, these would be test classes. Here we run them as async functions to demonstrate the pattern.

In [ ]:
async def test_purchase_flow():
    """Simulate a multi-turn purchase conversation."""
    async with echo_graph.session() as s:
        r1 = await s.chat("I want to buy a laptop")
        r2 = await s.chat("What's the price?")
        r3 = await s.chat("I'll take it")

        # Verify state accumulated across all turns
        assert r3.state["call_count"] == 3
        print(f"Purchase flow: 3 turns completed, call_count={r3.state['call_count']}")

        # In a real test, you'd evaluate with conversation metrics:
        # fe.score(
        #     actual_output=r3.response,
        #     expected_output="Order confirmed",
        #     input="I'll take it",
        #     history=[...],
        # )


async def test_support_flow():
    """Simulate a multi-turn support conversation."""
    async with echo_graph.session() as s:
        r1 = await s.chat("My order is late")
        r2 = await s.chat("Order #12345")
        r3 = await s.chat("Can I get a refund?")

        assert r3.state["call_count"] == 3
        print(f"Support flow: 3 turns completed, call_count={r3.state['call_count']}")


await test_purchase_flow()
await test_support_flow()
print("\nAll multi-turn flows passed.")

---

## Next Steps

| Notebook | What you'll learn |
|----------|------------------|
| [Harness Basics](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langgraph-testing/harness-basics.ipynb) | `.chat()`, `ChatResult`, hooks, auto-detection |
| [Node Testing & Mocking](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langgraph-testing/node-mocking.ipynb) | `.node().run()`, `mock()`, isolated testing patterns |